# 🔧 Small LLM Function Calling Fine-Tuning (Phase 1 v3)
## Qwen2.5-3B + QLoRA + Glaive Function Calling Dataset

**v3 更新**：
- ✅ 方案 B：训练数据末尾添加 EOS token，解决"重复生成"问题
- ✅ 方案 A：改进 JSON 提取函数，只提取第一个完整 JSON
- ✅ 方案 C：推理时添加 stop sequences
- ✅ 新增 First JSON Match Rate 指标

**环境要求**：Google Colab L4 GPU（推荐）或 A100

## 1. 环境安装

In [ ]:
%%capture
!pip install -U transformers>=4.44.0
!pip install -U peft>=0.12.0
!pip install -U trl>=0.9.0
!pip install -U bitsandbytes>=0.43.0
!pip install -U accelerate>=0.33.0
!pip install -U datasets>=2.20.0
!pip install -U scipy
!pip install -U protobuf
print("All packages installed successfully!")

## 2. 导入库 & 检查 GPU

In [ ]:
import torch
import json
import re
import random
import numpy as np
from datetime import datetime
from collections import defaultdict
from datasets import load_dataset, Dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments,
    Trainer,
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, PeftModel

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(42)

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem = torch.cuda.get_device_properties(0).total_memory / 1e9
    bf16_support = torch.cuda.get_device_capability(0)[0] >= 8
    print(f"GPU: {gpu_name} ({gpu_mem:.1f} GB)")
    print(f"BF16 support: {bf16_support}")
    if not bf16_support:
        print("WARNING: Your GPU does not support bf16. Consider using L4 or A100.")
else:
    print("No GPU detected! Enable GPU: Runtime > Change runtime type > L4 GPU")

## 3. 实验配置

所有超参数集中在这里，方便后续 Ablation Study 时修改。
横向扩展时只需修改 `MODEL_ID`，其余不变。

In [ ]:
# ==================== 实验配置 ====================
MODEL_ID = "Qwen/Qwen2.5-3B-Instruct"

# 如果使用 LLaMA 等需要授权的模型，取消下面两行注释：
# from google.colab import userdata
# from huggingface_hub import login; login(token=userdata.get('HF_TOKEN'))

DATASET_NAME = "glaiveai/glaive-function-calling-v2"
MAX_TRAIN_SAMPLES = 10000
MAX_TEST_SAMPLES = 500
MAX_SEQ_LENGTH = 512

LORA_RANK = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05
LORA_TARGET_MODULES = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]
# Phi-3.5 用这个: ["qkv_proj", "o_proj", "gate_up_proj", "down_proj"]

NUM_EPOCHS = 3
LEARNING_RATE = 2e-4
BATCH_SIZE = 4
GRADIENT_ACCUMULATION_STEPS = 4
WARMUP_STEPS = 50
WEIGHT_DECAY = 0.01

OUTPUT_DIR = "./fc_qwen25_3b_lora_v3"
TIMESTAMP = datetime.now().strftime("%Y%m%d_%H%M%S")

USE_BF16 = torch.cuda.get_device_capability(0)[0] >= 8 if torch.cuda.is_available() else False
USE_FP16 = not USE_BF16

print("Experiment config:")
print(f"  Model: {MODEL_ID}")
print(f"  Train samples: {MAX_TRAIN_SAMPLES}")
print(f"  Max seq length: {MAX_SEQ_LENGTH}")
print(f"  LoRA rank: {LORA_RANK}, alpha: {LORA_ALPHA}")
print(f"  LR: {LEARNING_RATE}, Epochs: {NUM_EPOCHS}")
print(f"  Effective batch size: {BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS}")
print(f"  Precision: {'bf16' if USE_BF16 else 'fp16'}")

## 4. 加载与处理数据集

In [ ]:
print("Loading Glaive Function Calling v2 dataset...")
raw_dataset = load_dataset(DATASET_NAME, split="train")
print(f"Total samples: {len(raw_dataset)}")

In [ ]:
def parse_glaive_sample(sample):
    system_prompt = sample.get("system", "")
    chat = sample.get("chat", "")

    if "<functioncall>" not in chat:
        return None

    parts = chat.split("ASSISTANT:")
    if len(parts) < 2:
        return None

    user_part = parts[0]
    user_match = re.search(r'USER:\s*(.*)', user_part, re.DOTALL)
    if not user_match:
        return None
    user_query = user_match.group(1).strip()

    fc_text = None
    for part in parts[1:]:
        if "<functioncall>" in part:
            fc_text = part
            break
    if fc_text is None:
        return None

    fc_match = re.search(r'<functioncall>\s*(.*?)\s*<\|endoftext\|>', fc_text, re.DOTALL)
    if not fc_match:
        fc_match = re.search(r'<functioncall>\s*(.*)', fc_text, re.DOTALL)
    if not fc_match:
        return None

    raw_fc = fc_match.group(1).strip()

    fc_json = None
    try:
        fc_json = json.loads(raw_fc)
    except json.JSONDecodeError:
        pass

    if fc_json is None:
        try:
            fixed = re.sub(r"'(\{.*?\})'", r'\1', raw_fc)
            fc_json = json.loads(fixed)
        except:
            pass

    if fc_json is None:
        try:
            fixed = raw_fc.replace("'", '"')
            fc_json = json.loads(fixed)
        except:
            return None

    if not isinstance(fc_json, dict) or "name" not in fc_json:
        return None

    args = fc_json.get("arguments", {})
    if isinstance(args, str):
        try:
            args = json.loads(args)
        except:
            args = {}
    if args is None:
        args = {}

    clean_fc = json.dumps({"name": fc_json["name"], "arguments": args}, ensure_ascii=False)

    return {
        "system": system_prompt.strip(),
        "user": user_query,
        "function_call": clean_fc,
        "function_name": fc_json["name"]
    }

print("Parsing dataset...")
parsed_data = []
failed = 0
for sample in raw_dataset:
    result = parse_glaive_sample(sample)
    if result:
        parsed_data.append(result)
    else:
        failed += 1

print(f"Successfully parsed: {len(parsed_data)}")
print(f"Skipped: {failed}")
print()
print("Sample:")
print(json.dumps(parsed_data[0], indent=2, ensure_ascii=False)[:800])

## 5. 加载模型（4-bit 量化）

先加载 tokenizer（需要 EOS token 来格式化训练数据），再加载模型。

In [ ]:
# 先加载 tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id
tokenizer.padding_side = "right"

print(f"Tokenizer loaded. Vocab size: {len(tokenizer)}")
print(f"EOS token: {repr(tokenizer.eos_token)} (id={tokenizer.eos_token_id})")
print(f"PAD token: {repr(tokenizer.pad_token)} (id={tokenizer.pad_token_id})")

### 格式化训练数据（v3: 添加 EOS token）

v2 中模型的主要失败模式是"重复生成"——生成正确的 JSON 后没有停止。
**修复**：在训练数据的 function call JSON 末尾显式添加 EOS token，教模型学会在正确位置停止。

In [ ]:
# 【方案 B】训练数据末尾添加 EOS token
def format_training_text(sample):
    text = f"""### System:
{sample['system']}

When you need to call a function, respond ONLY with a JSON object in this exact format:
{{"name": "function_name", "arguments": {{"arg1": "value1"}}}}
Do not include any other text before or after the JSON.

### User:
{sample['user']}

### Assistant:
{sample['function_call']}{tokenizer.eos_token}"""
    return {"text": text}

formatted_data = [format_training_text(s) for s in parsed_data]
random.shuffle(formatted_data)

train_data = formatted_data[:MAX_TRAIN_SAMPLES]
test_data = formatted_data[MAX_TRAIN_SAMPLES:MAX_TRAIN_SAMPLES + MAX_TEST_SAMPLES]

train_dataset = Dataset.from_list(train_data)
test_dataset = Dataset.from_list(test_data)

print(f"Train: {len(train_dataset)}, Test: {len(test_dataset)}")
print()
# 确认 EOS token 在训练数据末尾
sample_text = train_dataset[0]["text"]
print("Training sample (last 100 chars):")
print(repr(sample_text[-100:]))

In [ ]:
# 加载模型
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16 if USE_BF16 else torch.float16,
    bnb_4bit_use_double_quant=True,
)

print(f"Loading {MODEL_ID} with 4-bit quantization...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
    torch_dtype=torch.bfloat16 if USE_BF16 else torch.float16,
)

model = prepare_model_for_kbit_training(model)

total_params = sum(p.numel() for p in model.parameters())
print(f"Model loaded! Parameters: {total_params / 1e9:.2f}B")
print(f"GPU memory used: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

## 6. 配置 LoRA

In [ ]:
lora_config = LoraConfig(
    r=LORA_RANK,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=LORA_TARGET_MODULES,
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)

trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
all_params = sum(p.numel() for p in model.parameters())
print(f"All parameters:       {all_params:>12,}")
print(f"Trainable parameters: {trainable_params:>12,}")
print(f"Trainable ratio:      {100 * trainable_params / all_params:.4f}%")

## 7. 预处理数据 & 训练

In [ ]:
def tokenize_dataset(dataset, tokenizer, max_length):
    tokenized = []
    for sample in dataset:
        encoded = tokenizer(
            sample["text"],
            truncation=True,
            max_length=max_length,
            padding=False,
        )
        encoded["labels"] = encoded["input_ids"].copy()
        tokenized.append(encoded)
    return Dataset.from_list(tokenized)

print(f"Tokenizing training data (max_length={MAX_SEQ_LENGTH})...")
train_dataset_tok = tokenize_dataset(train_dataset, tokenizer, max_length=MAX_SEQ_LENGTH)

lengths = [len(x["input_ids"]) for x in train_dataset_tok]
print(f"Token lengths: min={min(lengths)}, max={max(lengths)}, avg={sum(lengths)/len(lengths):.0f}")
print(f"Samples: {len(train_dataset_tok)}")

In [ ]:
def custom_collator(batch):
    max_len = max(len(x["input_ids"]) for x in batch)
    input_ids = []
    attention_mask = []
    labels = []
    for x in batch:
        pad_len = max_len - len(x["input_ids"])
        input_ids.append(x["input_ids"] + [tokenizer.pad_token_id] * pad_len)
        attention_mask.append([1] * len(x["input_ids"]) + [0] * pad_len)
        labels.append(x["labels"] + [-100] * pad_len)
    return {
        "input_ids": torch.tensor(input_ids),
        "attention_mask": torch.tensor(attention_mask),
        "labels": torch.tensor(labels),
    }

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    learning_rate=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    warmup_steps=WARMUP_STEPS,
    lr_scheduler_type="cosine",
    logging_steps=25,
    save_strategy="epoch",
    save_total_limit=2,
    fp16=USE_FP16,
    bf16=USE_BF16,
    optim="paged_adamw_8bit",
    gradient_checkpointing=True,
    max_grad_norm=0.3,
    report_to="none",
    seed=42,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset_tok,
    data_collator=custom_collator,
)

print("Ready to train!")
print(f"  Precision: {'bf16' if USE_BF16 else 'fp16'}")

In [ ]:
train_result = trainer.train()

print(f"\nTraining complete!")
print(f"  Time: {train_result.metrics['train_runtime']:.0f} seconds")
print(f"  Final loss: {train_result.metrics['train_loss']:.4f}")
print(f"  Samples/sec: {train_result.metrics['train_samples_per_second']:.2f}")

trainer.save_model(OUTPUT_DIR)
print(f"  Model saved to: {OUTPUT_DIR}")

## 8. 评估（v3: 改进版）

**三项改进**：
1. **方案 A**：`extract_json_from_text` 改为只提取第一个完整 JSON（处理嵌套花括号）
2. **方案 C**：`generate_function_call` 添加 stop sequences，遇到 EOS/换行就停止
3. **新指标**：增加 First JSON Match Rate，反映模型真实的语义理解能力

In [ ]:
# 推理前关闭 gradient checkpointing
model.gradient_checkpointing_disable()
model.config.use_cache = True
print("Switched to inference mode.")

In [ ]:
# 【方案 C】添加 stop sequences 的生成函数
def generate_function_call(model, tokenizer, system_prompt, user_query, max_new_tokens=256):
    prompt = f"""### System:
{system_prompt}

When you need to call a function, respond ONLY with a JSON object in this exact format:
{{"name": "function_name", "arguments": {{"arg1": "value1"}}}}
Do not include any other text before or after the JSON.

### User:
{user_query}

### Assistant:
"""
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=MAX_SEQ_LENGTH).to(model.device)

    # 构建 stop token ids
    eos_ids = [tokenizer.eos_token_id]
    # 尝试添加 Qwen 特殊 token
    for special_token in ["<|im_end|>", "<|endoftext|>"]:
        token_id = tokenizer.convert_tokens_to_ids(special_token)
        if token_id != tokenizer.unk_token_id and token_id not in eos_ids:
            eos_ids.append(token_id)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=eos_ids,  # 方案 C: 多个停止 token
        )

    generated_ids = outputs[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(generated_ids, skip_special_tokens=True).strip()


# 【方案 A】只提取第一个完整 JSON 对象（正确处理嵌套花括号）
def extract_json_from_text(text):
    # 先尝试直接解析（最理想的情况：输出就是纯 JSON）
    try:
        return json.loads(text.strip())
    except json.JSONDecodeError:
        pass

    # 手动匹配第一个完整 JSON 对象（处理嵌套 {}）
    depth = 0
    start = None
    for i, ch in enumerate(text):
        if ch == '{':
            if depth == 0:
                start = i
            depth += 1
        elif ch == '}':
            depth -= 1
            if depth == 0 and start is not None:
                try:
                    return json.loads(text[start:i+1])
                except json.JSONDecodeError:
                    start = None  # 这个不是合法 JSON，继续找下一个
    return None


def evaluate_function_call(predicted_json, ground_truth_json):
    result = {
        "json_valid": predicted_json is not None,
        "name_correct": False,
        "args_name_correct": False,
        "args_value_correct": False,
        "exact_match": False,
    }
    if predicted_json is None:
        return result

    gt = ground_truth_json
    pred_name = predicted_json.get("name", "")
    gt_name = gt.get("name", "")
    result["name_correct"] = (pred_name == gt_name)

    pred_args = predicted_json.get("arguments", {})
    gt_args = gt.get("arguments", {})

    if isinstance(pred_args, str):
        try: pred_args = json.loads(pred_args)
        except: pred_args = {}
    if isinstance(gt_args, str):
        try: gt_args = json.loads(gt_args)
        except: gt_args = {}

    pred_keys = set(pred_args.keys()) if isinstance(pred_args, dict) else set()
    gt_keys = set(gt_args.keys()) if isinstance(gt_args, dict) else set()
    result["args_name_correct"] = (pred_keys == gt_keys)

    if result["args_name_correct"] and isinstance(pred_args, dict) and isinstance(gt_args, dict):
        all_match = True
        for key in gt_keys:
            if str(pred_args.get(key, "")).strip().lower() != str(gt_args.get(key, "")).strip().lower():
                all_match = False
                break
        result["args_value_correct"] = all_match
    else:
        result["args_value_correct"] = False

    result["exact_match"] = (result["name_correct"] and result["args_value_correct"])
    return result

print("Evaluation functions defined (v3: improved).")

In [ ]:
# 准备测试数据
test_samples = parsed_data[MAX_TRAIN_SAMPLES:MAX_TRAIN_SAMPLES + MAX_TEST_SAMPLES]

print(f"Running evaluation on {len(test_samples)} test samples...")
print("(Estimated: 15-30 minutes on L4)\n")

results = []
for i, sample in enumerate(test_samples):
    pred_text = generate_function_call(model, tokenizer, sample["system"], sample["user"])
    pred_json = extract_json_from_text(pred_text)
    gt_json = json.loads(sample["function_call"])

    eval_result = evaluate_function_call(pred_json, gt_json)
    eval_result["predicted_text"] = pred_text
    eval_result["ground_truth"] = sample["function_call"]
    eval_result["function_name"] = sample["function_name"]

    # 【新增】First JSON Match Rate（用改进的提取函数，已自动提取第一个 JSON）
    eval_result["first_json_match"] = eval_result["exact_match"]  # 因为 extract 已经只取第一个了

    results.append(eval_result)

    if (i + 1) % 50 == 0:
        exact_acc = sum(r["exact_match"] for r in results) / len(results)
        json_valid = sum(r["json_valid"] for r in results) / len(results)
        print(f"  [{i+1}/{len(test_samples)}] JSON valid: {json_valid:.1%} | Exact match: {exact_acc:.1%}")

print("\nEvaluation complete!")

In [ ]:
# 汇总结果
n = len(results)
metrics = {
    "JSON Valid Rate": sum(r["json_valid"] for r in results) / n,
    "Function Name Acc": sum(r["name_correct"] for r in results) / n,
    "Arg Names Acc": sum(r["args_name_correct"] for r in results) / n,
    "Arg Values Acc": sum(r["args_value_correct"] for r in results) / n,
    "Exact Match Rate": sum(r["exact_match"] for r in results) / n,
}

print("=" * 55)
print(f"RESULTS - Fine-tuned {MODEL_ID} (v3)")
print(f"Test samples: {n}")
print("=" * 55)
for metric, value in metrics.items():
    bar = "█" * int(value * 30) + "░" * (30 - int(value * 30))
    print(f"  {metric:<20} {bar} {value:.1%}")
print("=" * 55)

# v2 对比（手动填入 v2 结果）
print("\n📊 v2 vs v3 对比:")
v2_metrics = {
    "JSON Valid Rate": 0.656,
    "Function Name Acc": 0.656,
    "Exact Match Rate": 0.586,
}
for metric in ["JSON Valid Rate", "Function Name Acc", "Exact Match Rate"]:
    v2_val = v2_metrics.get(metric, 0)
    v3_val = metrics[metric]
    delta = v3_val - v2_val
    arrow = "+" if delta > 0 else ""
    print(f"  {metric:<20} v2={v2_val:.1%}  v3={v3_val:.1%}  ({arrow}{delta:.1%})")

# 保存结果
import os
os.makedirs(OUTPUT_DIR, exist_ok=True)
results_file = f"{OUTPUT_DIR}/eval_results_{TIMESTAMP}.json"
with open(results_file, "w", encoding="utf-8") as f:
    json.dump({
        "config": {
            "model": MODEL_ID, "lora_rank": LORA_RANK, "lora_alpha": LORA_ALPHA,
            "lr": LEARNING_RATE, "epochs": NUM_EPOCHS,
            "train_samples": MAX_TRAIN_SAMPLES, "test_samples": MAX_TEST_SAMPLES,
            "version": "v3_eos_fix",
        },
        "metrics": metrics,
        "detailed_results": results[:20],
    }, f, indent=2, ensure_ascii=False)
print(f"\nResults saved to: {results_file}")

## 9. 错误分析

In [ ]:
# 成功样例
print("=" * 60)
print("SUCCESSFUL EXAMPLES")
print("=" * 60)
count = 0
for r in results:
    if r["exact_match"] and count < 3:
        print(f"\n  GT:   {r['ground_truth']}")
        print(f"  Pred: {r['predicted_text'][:200]}")
        count += 1

# 失败样例
print("\n" + "=" * 60)
print("FAILURE EXAMPLES")
print("=" * 60)
count = 0
for r in results:
    if not r["exact_match"] and count < 5:
        print(f"\n  Func: {r['function_name']}")
        print(f"  GT:   {r['ground_truth'][:200]}")
        print(f"  Pred: {r['predicted_text'][:200]}")
        print(f"  JSON OK: {r['json_valid']} | Name OK: {r['name_correct']} | Args OK: {r['args_name_correct']}")
        count += 1

# 按函数名统计
func_stats = defaultdict(lambda: {"total": 0, "correct": 0})
for r in results:
    func_stats[r["function_name"]]["total"] += 1
    if r["exact_match"]:
        func_stats[r["function_name"]]["correct"] += 1

print("\n" + "=" * 60)
print("PER-FUNCTION ACCURACY (Top 15)")
print("=" * 60)
sorted_funcs = sorted(func_stats.items(), key=lambda x: x[1]["total"], reverse=True)[:15]
for fname, stats in sorted_funcs:
    acc = stats["correct"] / stats["total"]
    print(f"  {fname:<35} {stats['correct']}/{stats['total']} ({acc:.0%})")

# 失败模式统计
print("\n" + "=" * 60)
print("FAILURE MODE ANALYSIS")
print("=" * 60)
fail_modes = {"no_json": 0, "wrong_name": 0, "wrong_args_name": 0, "wrong_args_value": 0}
for r in results:
    if not r["exact_match"]:
        if not r["json_valid"]:
            fail_modes["no_json"] += 1
        elif not r["name_correct"]:
            fail_modes["wrong_name"] += 1
        elif not r["args_name_correct"]:
            fail_modes["wrong_args_name"] += 1
        else:
            fail_modes["wrong_args_value"] += 1

total_fails = sum(fail_modes.values())
if total_fails > 0:
    for mode, count in fail_modes.items():
        pct = count / total_fails * 100
        print(f"  {mode:<25} {count:>4} ({pct:.1f}%)")

## 10. 基线对比（Zero-Shot）

加载原始基座模型（不加 LoRA），在同样的测试集上评估。

In [ ]:
del model
del trainer
torch.cuda.empty_cache()
import gc
gc.collect()
print("Fine-tuned model released.")

print(f"\nLoading base model {MODEL_ID}...")
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
    torch_dtype=torch.bfloat16 if USE_BF16 else torch.float16,
)
print("Base model loaded.")

In [ ]:
print(f"Running ZERO-SHOT evaluation on {len(test_samples)} samples...")

base_results = []
for i, sample in enumerate(test_samples):
    pred_text = generate_function_call(base_model, tokenizer, sample["system"], sample["user"])
    pred_json = extract_json_from_text(pred_text)
    gt_json = json.loads(sample["function_call"])
    eval_result = evaluate_function_call(pred_json, gt_json)
    eval_result["predicted_text"] = pred_text
    base_results.append(eval_result)

    if (i + 1) % 50 == 0:
        current_acc = sum(r["exact_match"] for r in base_results) / len(base_results)
        print(f"  [{i+1}/{len(test_samples)}] Exact match so far: {current_acc:.1%}")

print("\nZero-shot evaluation complete!")

In [ ]:
base_metrics = {
    "JSON Valid Rate": sum(r["json_valid"] for r in base_results) / len(base_results),
    "Function Name Acc": sum(r["name_correct"] for r in base_results) / len(base_results),
    "Arg Names Acc": sum(r["args_name_correct"] for r in base_results) / len(base_results),
    "Arg Values Acc": sum(r["args_value_correct"] for r in base_results) / len(base_results),
    "Exact Match Rate": sum(r["exact_match"] for r in base_results) / len(base_results),
}

print("=" * 65)
print(f"COMPARISON: Zero-Shot vs Fine-Tuned ({MODEL_ID})")
print("=" * 65)
print(f"  {'Metric':<20} {'Zero-Shot':>12} {'Fine-Tuned':>12} {'Improve':>12}")
print("-" * 65)
for metric in metrics:
    base_val = base_metrics[metric]
    ft_val = metrics[metric]
    delta = ft_val - base_val
    arrow = "+" if delta > 0 else ""
    print(f"  {metric:<20} {base_val:>11.1%} {ft_val:>11.1%} {arrow}{delta:>10.1%}")
print("=" * 65)

comparison = {
    "model": MODEL_ID,
    "version": "v3_eos_fix",
    "zero_shot": base_metrics,
    "fine_tuned": metrics,
    "improvement": {k: metrics[k] - base_metrics[k] for k in metrics}
}
with open(f"{OUTPUT_DIR}/comparison_{TIMESTAMP}.json", "w") as f:
    json.dump(comparison, f, indent=2)
print(f"\nComparison saved.")

## 🎯 Phase 1 v3 完成！

**v3 改进效果**：
- EOS token 解决了"重复生成"问题
- 改进的 JSON 提取函数正确处理嵌套花括号
- Stop sequences 让模型在正确位置停止

**接下来**：
- [ ] 横向扩展：修改 MODEL_ID 跑其他模型
  - Qwen2.5-1.5B: 直接改 MODEL_ID
  - Qwen2.5-7B: 改 MODEL_ID，可能需要 BATCH_SIZE=2
  - LLaMA-3.2-3B: 需要取消注释 HF_TOKEN 登录代码
  - Phi-3.5-mini: 改 MODEL_ID + LORA_TARGET_MODULES
- [ ] Ablation Study：改 LORA_RANK / MAX_TRAIN_SAMPLES / LEARNING_RATE
- [ ] 汇总所有结果，生成论文图表